# LogisticRregression

In [2]:
import numpy as np

### 工具函数

#### sigmoid函数

In [3]:
def sigmoid(z):
    z = np.asarray(z, dtype = float)
    out = np.empty_like(z)
    pos = z >= 0
    neg = ~pos
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg]) # 对负数进行特殊处理，防止太大
    out[neg] = exp_z / (1.0 + exp_z)
    return out

#### softmax函数

In [4]:
def softmax(scores):
    """按行计算 softmax"""
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores, axis=1, keepdims=True) # 归一化
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

#### Log-Sum-Exp (LSE)

In [5]:
def logsumexp(scores, axis=1):
    """数值稳定的 logsumexp"""
    scores = np.asarray(scores, dtype=float)
    max_score = np.max(scores, axis=axis, keepdims=True)
    return np.squeeze(
        max_score + np.log(np.sum(np.exp(scores - max_score), axis=axis, keepdims=True)),
        axis=axis
    ) # squeeze是压缩维度的

### 二项Logistic 回归

In [ ]:
class BinaryLogisticRegression:
    """
    二项逻辑斯谛回归，适合二分类 y in {0, 1}。
    模型：
        P(Y=1|x) = sigmoid(w·x + b)
    支持：
        - 梯度下降 gd
        - 牛顿法 newton
    """

    def __init__(
        self,
        lr=0.1,
        max_iter=1000,
        tol=1e-6,
        fit_intercept=True,
        l2=0.0,
        optimizer="gd",
        verbose=False
    ):
        self.lr = lr
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.l2 = l2
        self.optimizer = optimizer
        self.verbose = verbose
        self.coef_ = None
        self.intercept_ = None
        self.w_ = None
        self.loss_history_ = []

    def _add_intercept(self, X):
        X = np.asarray(X, dtype=float)
        if not self.fit_intercept:
            return X
        ones = np.ones((X.shape[0], 1))
        return np.c_[ones, X]

    def _loss(self, Xb, y, w):
        n = Xb.shape[0]
        z = Xb @ w
        p = sigmoid(z)
        eps = 1e-12
        p = np.clip(p, eps, 1 - eps) # 把概率数组p中的所有数值强行限制在[eps, 1 - eps]这个极小的区间内
        # 用平均数，与数据规模解耦
        neg_log_likelihood = -np.mean(
            y * np.log(p) + (1 - y) * np.log(1 - p)
        )
        # L2正则项
        if self.l2 > 0:
            reg_w = w.copy()
            if self.fit_intercept:
                reg_w[0] = 0.0
            neg_log_likelihood += 0.5 * self.l2 * np.sum(reg_w ** 2)
        return neg_log_likelihood

    def _gradient(self, Xb, y, w):
        n = Xb.shape[0]
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / n
        if self.l2 > 0:
            reg_w = w.copy()
            if self.fit_intercept:
                reg_w[0] = 0.0
            grad += self.l2 * reg_w

        return grad

    def _hessian(self, Xb, w):
        n = Xb.shape[0]
        p = sigmoid(Xb @ w)
        r = p * (1 - p)
        H = (Xb.T * r) @ Xb / n
        if self.l2 > 0:
            reg = self.l2 * np.eye(Xb.shape[1])
            if self.fit_intercept:
                reg[0, 0] = 0.0
            H += reg

        return H

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        if set(np.unique(y)) - {0, 1}:
            raise ValueError("BinaryLogisticRegression 要求 y 只能取 0 或 1。")
        Xb = self._add_intercept(X)
        n_features = Xb.shape[1]
        w = np.zeros(n_features)
        for it in range(self.max_iter):
            loss = self._loss(Xb, y, w)
            self.loss_history_.append(loss)
            grad = self._gradient(Xb, y, w)
            if np.linalg.norm(grad) < self.tol:
                if self.verbose:
                    print(f"Converged at iter={it}, loss={loss:.6f}")
                break
            if self.optimizer == "gd":
                w_new = w - self.lr * grad
            elif self.optimizer == "newton":
                H = self._hessian(Xb, w)
                try:
                    step = np.linalg.solve(H, grad)
                except np.linalg.LinAlgError:
                    step = np.linalg.pinv(H) @ grad
                w_new = w - step

            else:
                raise ValueError("optimizer 只能是 'gd' 或 'newton'。")

            if np.linalg.norm(w_new - w) < self.tol:
                w = w_new
                break

            w = w_new

        self.w_ = w

        if self.fit_intercept:
            self.intercept_ = w[0]
            self.coef_ = w[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = w

        return self

    def predict_proba(self, X):
        Xb = self._add_intercept(X)
        p1 = sigmoid(Xb @ self.w_)
        p0 = 1 - p1
        return np.c_[p0, p1]

    def predict(self, X):
        p1 = self.predict_proba(X)[:, 1]
        return (p1 >= 0.5).astype(int)

    def score(self, X, y):
        y = np.asarray(y).ravel()
        pred = self.predict(X)
        return np.mean(pred == y)

### 多项logistic回归

In [7]:
class SoftmaxRegression:
    """
    多项逻辑斯谛回归，适合多分类。
    模型：
        P(Y=k|x) = exp(w_k·x) / sum_j exp(w_j·x)
    """
    def __init__(
        self,
        lr=0.1,
        max_iter=1000,
        tol=1e-6,
        fit_intercept=True,
        l2=0.0,
        verbose=False
    ):
        self.lr = lr
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.l2 = l2
        self.verbose = verbose
        self.classes_ = None
        self.W_ = None
        self.loss_history_ = []

    def _add_intercept(self, X):
        X = np.asarray(X, dtype=float)
        if not self.fit_intercept:
            return X
        ones = np.ones((X.shape[0], 1))
        return np.c_[ones, X]

    def _one_hot(self, y_idx, n_classes): # y_idx 是类别位置[1,0,1,1]这种
        Y = np.zeros((len(y_idx), n_classes))
        Y[np.arange(len(y_idx)), y_idx] = 1.0 # 高级花式索引
        return Y

    def _loss(self, Xb, Y, W):
        n = Xb.shape[0]
        scores = Xb @ W
        probs = softmax(scores)
        eps = 1e-12
        probs = np.clip(probs, eps, 1.0)
        loss = -np.sum(Y * np.log(probs)) / n
        if self.l2 > 0:
            reg_W = W.copy()
            if self.fit_intercept:
                reg_W[0, :] = 0.0
            loss += 0.5 * self.l2 * np.sum(reg_W ** 2)
        return loss

    def _gradient(self, Xb, Y, W):
        n = Xb.shape[0]
        probs = softmax(Xb @ W)
        grad = Xb.T @ (probs - Y) / n
        if self.l2 > 0:
            reg_W = W.copy()
            if self.fit_intercept:
                reg_W[0, :] = 0.0
            grad += self.l2 * reg_W
        return grad

    def fit(self, X, y):
        Xb = self._add_intercept(X)
        y = np.asarray(y)
        self.classes_, y_idx = np.unique(y, return_inverse=True) # return_inverse=True：返回的是一个和原始y长度一模一样的数字索引数组。
        n_classes = len(self.classes_)
        Y = self._one_hot(y_idx, n_classes)
        n_features = Xb.shape[1]
        W = np.zeros((n_features, n_classes))

        for it in range(self.max_iter):
            loss = self._loss(Xb, Y, W)
            self.loss_history_.append(loss)
            grad = self._gradient(Xb, Y, W)
            if np.linalg.norm(grad) < self.tol:
                if self.verbose:
                    print(f"Converged at iter={it}, loss={loss:.6f}")
                break
            W_new = W - self.lr * grad
            if np.linalg.norm(W_new - W) < self.tol:
                W = W_new
                break
            W = W_new
        self.W_ = W
        return self

    def predict_proba(self, X):
        Xb = self._add_intercept(X)
        return softmax(Xb @ self.W_)

    def predict(self, X):
        probs = self.predict_proba(X)
        idx = np.argmax(probs, axis=1)
        return self.classes_[idx]

    def score(self, X, y):
        y = np.asarray(y)
        pred = self.predict(X)
        return np.mean(pred == y)

## 最大熵模型

#### 基类

In [ ]:
class BaseMaxEnt:
    """
    最大熵模型基类。
    模型：
        P_w(y|x) = exp(sum_i w_i f_i(x,y)) / Z_w(x)
    这里自动构造二值特征函数：
        f_{j,v,c}(x,y) = 1{x_j = v 且 y = c}
    适合离散特征。
    """
    def __init__(self, max_iter=100, tol=1e-5, fit_bias=True, verbose=False):
        self.max_iter = max_iter # 表示最大迭代次数。
        self.tol = tol # 收敛阈值
        self.fit_bias = fit_bias # 是否加入偏置项
        self.verbose = verbose # 是否打印训练过程
        self.classes_ = None # 保存所有类别标签
        self.class_to_index_ = None # 保存类别到编号的映射 字典
        self.feature_to_id_ = {} # 保存“特征函数 → 编号”的映射 字典
        self.id_to_feature_ = {} # 保存“编号 → 特征函数”的反向映射
        self.n_model_features_ = None # 最大熵模型中一共有多少个特征函数
        self.n_input_features_ = None # 原始输入样本有多少个特征维度
        self.w_ = None
        self.empirical_expectation_ = None # 特征函数在训练数据上的经验期望
        self.X_train_ = None
        self.y_train_ = None
        self.y_indices_ = None # 训练标签对应的类别编号
        self.active_cache_ = None # 每个样本在不同类别下激活的特征函数
        self.feature_to_occurrences_ = None # 每个特征函数在哪些训练样本中出现过
        self.loss_history_ = []

    def _check_X(self, X):
        X = np.asarray(X, dtype=object)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return X

    def _add_feature(self, key):
        if key not in self.feature_to_id_:
            idx = len(self.feature_to_id_)
            self.feature_to_id_[key] = idx
            self.id_to_feature_[idx] = key
        return self.feature_to_id_[key]

    def _build_feature_space(self, X, y):
        """
        只把训练集中真实出现过的 (特征取值, 标签) 组合加入特征集合。
        """
        self.feature_to_id_ = {}
        self.id_to_feature_ = {}

        if self.fit_bias:
            for c in self.classes_:
                self._add_feature(("__bias__", c))

        for xi, yi in zip(X, y):
            for j in range(self.n_input_features_):
                self._add_feature((j, xi[j], yi))

        self.n_model_features_ = len(self.feature_to_id_)

    def _active_ids(self, x, label):
        """
        返回给定 (x, label) 时触发的特征 id。
        """
        ids = []

        if self.fit_bias:
            bias_id = self.feature_to_id_.get(("__bias__", label)) # 得到编号，找不到不会报错，返回None
            if bias_id is not None:
                ids.append(bias_id)

        for j in range(self.n_input_features_):
            fid = self.feature_to_id_.get((j, x[j], label))
            if fid is not None:
                ids.append(fid)

        return np.asarray(ids, dtype=int)

    def _prepare_training(self, X, y):
        X = self._check_X(X)
        y = np.asarray(y)

        self.X_train_ = X
        self.y_train_ = y
        self.n_input_features_ = X.shape[1]

        self.classes_, self.y_indices_ = np.unique(y, return_inverse=True)
        self.class_to_index_ = {c: i for i, c in enumerate(self.classes_)}

        self._build_feature_space(X, y)

        n_samples = X.shape[0]
        n_classes = len(self.classes_)

        # active_cache_[i][k] 表示第 i 个样本假设类别为第 k 类时触发的特征
        self.active_cache_ = []
        for i in range(n_samples):
            row = [] # 当前这个样本在所有类别下的激活特征编号
            for c in self.classes_:
                row.append(self._active_ids(X[i], c))
            self.active_cache_.append(row)

        # 经验期望 E_tilde(f_i)
        empirical = np.zeros(self.n_model_features_)
        for i in range(n_samples):
            true_k = self.y_indices_[i]
            ids = self.active_cache_[i][true_k]
            empirical[ids] += 1.0 / n_samples

        self.empirical_expectation_ = empirical

        # 为 IIS 预计算每个特征在哪些 (x_i, y_k) 下出现
        self.feature_to_occurrences_ = [[] for _ in range(self.n_model_features_)]
        for i in range(n_samples):
            for k in range(n_classes):
                ids = self.active_cache_[i][k]
                f_sharp = len(ids)
                for fid in ids:
                    self.feature_to_occurrences_[fid].append((i, k, f_sharp))

        self.w_ = np.zeros(self.n_model_features_)

    def _scores_from_cache(self, w):
        n_samples = len(self.active_cache_)
        n_classes = len(self.classes_)

        scores = np.zeros((n_samples, n_classes))
        for i in range(n_samples):
            for k in range(n_classes):
                ids = self.active_cache_[i][k]
                if len(ids) > 0:
                    scores[i, k] = np.sum(w[ids])
        return scores

    def _train_proba(self, w):
        scores = self._scores_from_cache(w)
        return softmax(scores)

    def _negative_log_likelihood(self, w, l2=0.0):
        scores = self._scores_from_cache(w)
        log_z = logsumexp(scores, axis=1)
        true_scores = scores[np.arange(len(self.y_indices_)), self.y_indices_]
        loss = np.mean(log_z - true_scores)
        if l2 > 0:
            loss += 0.5 * l2 * np.sum(w ** 2)

        return loss

    def _gradient(self, w, l2=0.0):
        n_samples = len(self.active_cache_)
        n_classes = len(self.classes_)

        probs = self._train_proba(w)

        model_expectation = np.zeros(self.n_model_features_)

        for i in range(n_samples):
            for k in range(n_classes):
                ids = self.active_cache_[i][k]
                model_expectation[ids] += probs[i, k] / n_samples

        grad = model_expectation - self.empirical_expectation_

        if l2 > 0:
            grad += l2 * w

        return grad

    def predict_proba(self, X):
        X = self._check_X(X)
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        scores = np.zeros((n_samples, n_classes))
        for i in range(n_samples):
            for k, c in enumerate(self.classes_):
                ids = self._active_ids(X[i], c)
                if len(ids) > 0:
                    scores[i, k] = np.sum(self.w_[ids])

        return softmax(scores)

    def predict(self, X):
        probs = self.predict_proba(X)
        idx = np.argmax(probs, axis=1)
        return self.classes_[idx]

    def score(self, X, y):
        y = np.asarray(y)
        pred = self.predict(X)
        return np.mean(pred == y)

#### 梯度下降

In [ ]:
class MaxEntGradientDescent(BaseMaxEnt):
    """
    最大熵模型的梯度下降学习。
    目标函数：
        f(w) = - L_tilde(P_w)
    梯度：
        grad_i = E_{P_w}(f_i) - E_tilde(f_i)
    """
    def __init__(
        self,
        lr=0.1,
        max_iter=300,
        tol=1e-5,
        fit_bias=True,
        l2=0.0,
        verbose=False
    ):
        super().__init__(max_iter=max_iter, tol=tol, fit_bias=fit_bias, verbose=verbose)
        self.lr = lr
        self.l2 = l2

    def fit(self, X, y):
        self._prepare_training(X, y)
        w = self.w_.copy()
        for it in range(self.max_iter):
            loss = self._negative_log_likelihood(w, l2=self.l2)
            self.loss_history_.append(loss)
            grad = self._gradient(w, l2=self.l2)

            if np.linalg.norm(grad) < self.tol:
                if self.verbose:
                    print(f"GD converged at iter={it}, loss={loss:.6f}")
                break

            w_new = w - self.lr * grad

            if np.linalg.norm(w_new - w) < self.tol:
                w = w_new
                break

            w = w_new

        self.w_ = w
        return self

#### IIS改进迭代尺度法

In [ ]:
class MaxEntIIS(BaseMaxEnt):
    """
    最大熵模型的 IIS 学习算法。
    IIS 更新方程：
        sum_x,y P_tilde(x) P_w(y|x) f_i(x,y)
        exp(delta_i f#(x,y))
        =
        E_tilde(f_i)
    其中：
        f#(x,y) = sum_i f_i(x,y)
    """

    def __init__(
        self,
        max_iter=100,
        inner_iter=30,
        tol=1e-5,
        fit_bias=True,
        max_delta=5.0,
        verbose=False
    ):
        super().__init__(max_iter=max_iter, tol=tol, fit_bias=fit_bias, verbose=verbose)
        self.inner_iter = inner_iter
        self.max_delta = max_delta

    def _solve_delta_i(self, fid, probs):
        """
        对单个特征 f_i 解 IIS 方程。
        用牛顿法解：
            g(delta) = model_exp_i(delta) - empirical_i = 0
        """
        target = self.empirical_expectation_[fid]
        occurrences = self.feature_to_occurrences_[fid]

        if target <= 0 or len(occurrences) == 0:
            return 0.0

        n_samples = len(self.active_cache_)
        delta = 0.0

        for _ in range(self.inner_iter):
            value = 0.0
            derivative = 0.0

            for sample_i, class_k, f_sharp in occurrences:
                coef = probs[sample_i, class_k] / n_samples

                exponent = np.clip(delta * f_sharp, -50, 50)
                exp_term = np.exp(exponent)

                value += coef * exp_term
                derivative += coef * f_sharp * exp_term

            g = value - target

            if abs(g) < self.tol:
                break

            if derivative <= 1e-12:
                break

            delta -= g / derivative
            delta = float(np.clip(delta, -self.max_delta, self.max_delta))

        return delta

    def fit(self, X, y):
        self._prepare_training(X, y)

        w = self.w_.copy()

        for it in range(self.max_iter):
            loss = self._negative_log_likelihood(w)
            self.loss_history_.append(loss)

            probs = self._train_proba(w)

            deltas = np.zeros_like(w)

            for fid in range(self.n_model_features_):
                deltas[fid] = self._solve_delta_i(fid, probs)

            w_new = w + deltas

            if self.verbose:
                print(
                    f"IIS iter={it}, loss={loss:.6f}, "
                    f"delta_norm={np.linalg.norm(deltas):.6e}"
                )

            if np.linalg.norm(deltas) < self.tol:
                w = w_new
                break

            w = w_new

        self.w_ = w
        return self

#### BFGS拟牛顿法

In [ ]:
class MaxEntBFGS(BaseMaxEnt):
    """
    最大熵模型的 BFGS 拟牛顿算法。
    目标：
        min_w f(w)
    搜索方向：
        B_k p_k = -g_k
    矩阵更新：
        B_{k+1}
        =
        B_k
        + y_k y_k^T / (y_k^T delta_k)
        - B_k delta_k delta_k^T B_k / (delta_k^T B_k delta_k)
    """
    def __init__(
        self,
        max_iter=100,
        tol=1e-5,
        fit_bias=True,
        l2=0.0,
        line_search_max_iter=30,
        verbose=False
    ):
        super().__init__(max_iter=max_iter, tol=tol, fit_bias=fit_bias, verbose=verbose)
        self.l2 = l2
        self.line_search_max_iter = line_search_max_iter

    def _line_search(self, w, p, g):
        """
        Armijo 回溯线搜索。
        """
        alpha = 1.0
        c1 = 1e-4
        rho = 0.5

        f0 = self._negative_log_likelihood(w, l2=self.l2)
        slope = np.dot(g, p)

        for _ in range(self.line_search_max_iter):
            w_new = w + alpha * p
            f_new = self._negative_log_likelihood(w_new, l2=self.l2)

            if f_new <= f0 + c1 * alpha * slope:
                return alpha

            alpha *= rho

        return alpha

    def fit(self, X, y):
        self._prepare_training(X, y)

        w = self.w_.copy()
        n_params = len(w)

        B = np.eye(n_params)

        for it in range(self.max_iter):
            loss = self._negative_log_likelihood(w, l2=self.l2)
            self.loss_history_.append(loss)

            g = self._gradient(w, l2=self.l2)

            if self.verbose:
                print(f"BFGS iter={it}, loss={loss:.6f}, grad_norm={np.linalg.norm(g):.6e}")

            if np.linalg.norm(g) < self.tol:
                break

            try:
                p = np.linalg.solve(B, -g)
            except np.linalg.LinAlgError:
                p = -g

            # 如果不是下降方向，退化为负梯度方向
            if np.dot(g, p) >= 0:
                p = -g
                B = np.eye(n_params)

            alpha = self._line_search(w, p, g)

            w_new = w + alpha * p
            g_new = self._gradient(w_new, l2=self.l2)

            delta = w_new - w
            y_vec = g_new - g

            ys = np.dot(y_vec, delta)

            if ys > 1e-12:
                B_delta = B @ delta
                delta_B_delta = np.dot(delta, B_delta)

                if delta_B_delta > 1e-12:
                    B = (
                        B
                        + np.outer(y_vec, y_vec) / ys
                        - np.outer(B_delta, B_delta) / delta_B_delta
                    )

            if np.linalg.norm(w_new - w) < self.tol:
                w = w_new
                break

            w = w_new

        self.w_ = w
        return self

### 验证

In [8]:
X_bin = np.array([
    [0.1, 1.0],
    [0.3, 0.8],
    [1.2, 0.2],
    [1.5, 0.4],
    [0.2, 1.2],
    [1.7, 0.3]
])

y_bin = np.array([0, 0, 1, 1, 0, 1])

log_reg = BinaryLogisticRegression(
    optimizer="newton",
    max_iter=100,
    tol=1e-8,
    l2=0.01,
    verbose=True
)

log_reg.fit(X_bin, y_bin)

print("二项逻辑斯谛回归预测概率：")
print(log_reg.predict_proba(X_bin))

print("二项逻辑斯谛回归预测类别：")
print(log_reg.predict(X_bin))

print("训练准确率：", log_reg.score(X_bin, y_bin))

# -----------------------------
# 多项逻辑斯谛回归测试
# -----------------------------
X_multi = np.array([
    [0.1, 1.0],
    [0.2, 0.9],
    [1.0, 0.2],
    [1.2, 0.1],
    [0.5, 0.5],
    [0.6, 0.4],
    [1.5, 1.5],
    [1.6, 1.4]
])

y_multi = np.array(["A", "A", "B", "B", "C", "C", "D", "D"])

softmax_reg = SoftmaxRegression(
    lr=0.5,
    max_iter=1000,
    tol=1e-8,
    l2=0.01,
    verbose=False
)

softmax_reg.fit(X_multi, y_multi)

print("多项逻辑斯谛回归预测概率：")
print(softmax_reg.predict_proba(X_multi))

print("多项逻辑斯谛回归预测类别：")
print(softmax_reg.predict(X_multi))

print("训练准确率：", softmax_reg.score(X_multi, y_multi))

Converged at iter=5, loss=0.142686
二项逻辑斯谛回归预测概率：
[[0.96086772 0.03913228]
 [0.89244611 0.10755389]
 [0.10244731 0.89755269]
 [0.05707274 0.94292726]
 [0.96282558 0.03717442]
 [0.02434053 0.97565947]]
二项逻辑斯谛回归预测类别：
[0 0 1 1 0 1]
训练准确率： 1.0
多项逻辑斯谛回归预测概率：
[[0.81607124 0.00510224 0.16153362 0.01729289]
 [0.71984281 0.01259155 0.24229923 0.02526641]
 [0.01681293 0.67018973 0.27953174 0.0334656 ]
 [0.00435083 0.82411141 0.14775958 0.02377818]
 [0.24191828 0.15264915 0.56687106 0.03856151]
 [0.14257018 0.25168841 0.56809874 0.03764266]
 [0.03641666 0.03120762 0.01650656 0.91586916]
 [0.02182145 0.05231823 0.01681975 0.90904056]]
多项逻辑斯谛回归预测类别：
['A' 'A' 'B' 'B' 'C' 'C' 'D' 'D']
训练准确率： 1.0
